# 01 — Extract latitude–depth sections and save them in compact form

**Run once, on the server** (needs the `*_masked*.nc` files and `AMOC_{MODEL}.nc`).

For each model × variable (`thetao`, `so`) × longitude (10, 20, 30, 40, 60°W), the latitude–depth section of every
member is extracted. The geometry is the same as in `Paper_notebook.ipynb`: for each grid row, the nearest valid
ocean point within ±10° of the target longitude. The notebook saves:

```
data/{MODEL}_thetao_sections.nc   anomalies (member, year, lat, depth) + per-member climatology
data/{MODEL}_so_sections.nc
data/{MODEL}_amoc.nc              AMOC series (26N, 45N)
```

Size reduction:
- only the 5 sections are kept, not the 3D fields;
- `LAT_RES`: grid rows averaged onto a regular latitude grid (1° by default; `None` = native rows);
- `DEPTH_MAX`: maximum depth kept;
- anomalies stored as float32, with zlib compression and quantisation (`LSD` significant decimals);
- `MAX_MEMBERS`: optional cap on the number of members.

Select models / variables with `MODELS` and `VARS`. Existing output files are skipped, so models can be processed one at a time.

`02_EOF_AMOC_reconstruction.ipynb` then runs **only** from these files.

In [ ]:
import os, glob, re, warnings
import numpy as np
import xarray as xr

warnings.filterwarnings("ignore")
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# ============================================================
# CONFIG
# ============================================================
MODELS = ["CESM2", "EC-Earth3", "IPSL-CM6A-LR", "MPI-ESM1-2-LR"]
VARS = ["thetao", "so"]
TARGET_LONS = [-10.0, -20.0, -30.0, -40.0, -60.0]
LON_WINDOW = 10.0            # degrees (nearest valid ocean point within +-LON_WINDOW)
DEPTH_MAX = 5000.0           # m
YEAR_START, YEAR_END = 1850, 2014
TRAIN_END_YEAR = 1989        # 85 % of 1850-2014 (same split as the repo)

LAT_RES = 1.0                # regular latitude grid (deg) ; None = native grid rows
MAX_MEMBERS = None           # e.g. 10 to cap the ensemble size
LSD = {"thetao": 3, "so": 4} # significant decimals kept (0.001 degC, 0.0001 psu)

IN_DIR_TPL = "/data/projects/nckf/frekle/CMIP6_data/{model}/{var}/masked/"
AMOC_FILE_TPL = "/data/users/frekle/AMOC_analysis/AMOC_{model}.nc"
OUT_DIR = "./data"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ============================================================
# HELPERS (geometry logic identical to Paper_notebook.ipynb)
# ============================================================
def lon_tag(lon):
    return f"{'W' if lon < 0 else 'E'}{abs(lon):04.1f}".replace(".", "p")

LON_TAGS = [lon_tag(l) for l in TARGET_LONS]

def wrap_lon(lon):
    return ((np.asarray(lon) + 180) % 360) - 180

def member_label(path):
    m = re.search(r"_(r\d+i\d+p\d+f\d+)", os.path.basename(path))
    return m.group(1) if m else os.path.basename(path)

def standardize_latlon(ds):
    """Rename horizontal dims to (y, x) and provide 2D lat2d / lon2d."""
    lat_v = next(v for v in ["lat", "latitude", "nav_lat"] if v in ds)
    lon_v = next(v for v in ["lon", "longitude", "nav_lon"] if v in ds)
    lat, lon = ds[lat_v], ds[lon_v]
    if lat.ndim == 1 and lon.ndim == 1:
        lon2d, lat2d = np.meshgrid(lon.values, lat.values)
        ds = ds.drop_vars([lat_v, lon_v]).rename_dims({lat.dims[0]: "y", lon.dims[0]: "x"})
        ds["lat2d"] = (("y", "x"), lat2d)
        ds["lon2d"] = (("y", "x"), lon2d)
        return ds
    ds = ds.swap_dims({lat.dims[0]: "y", lat.dims[1]: "x"})
    return ds.rename({lat_v: "lat2d", lon_v: "lon2d"})

def get_depth(da, ds):
    for name in ["olevel", "depth", "lev"]:
        if name in da.dims:
            z = np.asarray(ds[name].values, float)
            if np.nanmax(z) > 10000:   # CESM: cm -> m
                z = z / 100.0
            return name, z
    raise KeyError("No vertical coordinate found")

def layer_thickness(z):
    z = np.asarray(z, float)
    dz = np.empty_like(z)
    dz[1:-1] = 0.5 * (z[2:] - z[:-2])
    dz[0], dz[-1] = z[1] - z[0], z[-1] - z[-2]
    return np.abs(dz)

def section_columns(lon2d, lat2d, valid_xy, target_lon, window=LON_WINDOW):
    """For every grid row y: nearest valid ocean column within +-window of target_lon.
    Returns (rows, cols, lat) sorted by latitude."""
    d = np.abs(wrap_lon(lon2d) - wrap_lon(target_lon))
    d = np.minimum(d, 360 - d)
    d = np.where(valid_xy & (d <= window), d, np.inf)
    rows = np.where(np.isfinite(d.min(axis=1)))[0]
    cols = d[rows].argmin(axis=1)
    lat = lat2d[rows, cols]
    o = np.argsort(lat)
    return rows[o], cols[o], lat[o]

def bin_to_lat_grid(arr, lat, res):
    """Average rows (axis -2) into regular latitude bins of width res."""
    if res is None:
        return arr, lat
    b = np.floor((lat + 90.0) / res).astype(int)
    ub = np.unique(b)
    out = np.empty(arr.shape[:-2] + (len(ub), arr.shape[-1]), np.float32)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        for i, bb in enumerate(ub):
            out[..., i, :] = np.nanmean(arr[..., b == bb, :], axis=-2)
    return out, -90.0 + (ub + 0.5) * res

def extract_years(time):
    try:
        return xr.DataArray(time).dt.year.values.astype(int)
    except Exception:
        return np.array([int(str(t)[:4]) for t in np.asarray(time)])

def read_member(path, var, geom=None):
    """Read one member file once and extract the 5 sections -> {lon_tag: (year, row, depth)}."""
    ds = standardize_latlon(xr.open_dataset(path))
    da = ds[var].sel(time=slice(f"{YEAR_START}-01-01", f"{YEAR_END}-12-31"))
    zdim, z = get_depth(da, ds)
    kz = np.where(z <= DEPTH_MAX)[0]
    da = da.isel({zdim: kz})

    if geom is None:  # geometry computed once per model/var (same grid & mask for all members)
        valid = np.isfinite(da.isel(time=0)).any(zdim).values
        geom = {lt: section_columns(ds["lon2d"].values, ds["lat2d"].values, valid, lon)
                for lon, lt in zip(TARGET_LONS, LON_TAGS)}
        geom["z"], geom["dz"] = z[kz], layer_thickness(z)[kz]

    years = extract_years(da["time"].values)
    annual = len(np.unique(years)) == len(years)
    out = {}
    for lt in LON_TAGS:
        rows, cols, _ = geom[lt]
        ucols = np.unique(cols)
        sub = da.isel(x=ucols)
        if not annual:  # monthly input -> annual means
            sub = sub.groupby("time.year").mean("time").rename(year="time")
        v = sub.transpose("time", "y", "x", zdim).values.astype(np.float32)
        out[lt] = v[:, rows, np.searchsorted(ucols, cols), :]   # (year, row, depth)
    if not annual:
        years = np.unique(years)
    ds.close()
    return years, out, geom

In [ ]:
# ============================================================
# EXTRACTION + SAVE  (one file per model and variable)
# ============================================================
for model in MODELS:
    for var in VARS:
        outfile = os.path.join(OUT_DIR, f"{model}_{var}_sections.nc")
        if os.path.exists(outfile):
            print("exists, skip:", outfile)
            continue
        files = sorted(glob.glob(os.path.join(IN_DIR_TPL.format(model=model, var=var), f"*{var}*_masked*.nc")))
        files = files[:MAX_MEMBERS] if MAX_MEMBERS else files
        if not files:
            print(f"!! no files for {model} {var}")
            continue

        geom, labels, yrs, secs = None, [], [], {lt: [] for lt in LON_TAGS}
        for i, f in enumerate(files, 1):
            years, out, geom = read_member(f, var, geom)
            labels.append(member_label(f)); yrs.append(years)
            for lt in LON_TAGS:
                secs[lt].append(out[lt])
            print(f"{model} {var}: member {i}/{len(files)} {labels[-1]}", end="\r")

        common = yrs[0]
        for y in yrs[1:]:
            common = np.intersect1d(common, y)
        train = common <= TRAIN_END_YEAR

        coords = {"member": np.array(labels, dtype=str), "year": common, "depth": geom["z"]}
        dvars = {"dz": (("depth",), geom["dz"])}
        for lt in LON_TAGS:
            arr = np.stack([s[np.isin(y, common)] for s, y in zip(secs[lt], yrs)])  # (member, year, row, depth)
            arr, lat = bin_to_lat_grid(arr, geom[lt][2], LAT_RES)
            clim = np.nanmean(arr[:, train], axis=1)                                   # (member, lat, depth)
            dvars[f"{var}_{lt}"] = (("member", "year", f"lat_{lt}", "depth"), arr - clim[:, None])
            dvars[f"{var}_{lt}_clim"] = (("member", f"lat_{lt}", "depth"), clim)
            coords[f"lat_{lt}"] = lat
        secs = None  # free memory

        ds_out = xr.Dataset(dvars, coords=coords, attrs=dict(
            MODEL=model, VAR=var, CONTENT="anomalies w.r.t. member mean over years <= TRAIN_END_YEAR; add *_clim for raw values",
            TRAIN_END_YEAR=TRAIN_END_YEAR, LON_WINDOW_DEG=LON_WINDOW, LAT_RES=str(LAT_RES), DEPTH_MAX=DEPTH_MAX))
        enc = {k: dict(zlib=True, complevel=5, dtype="float32", least_significant_digit=LSD[var])
               for k in ds_out.data_vars if k.startswith(var)}
        ds_out.to_netcdf(outfile, encoding=enc)
        print(f"\n✓ {outfile}  ({os.path.getsize(outfile)/1e6:.0f} MB, {len(labels)} members, "
              f"{common[0]}-{common[-1]})")

In [ ]:
# ============================================================
# AMOC series (already computed on the server; just copied with a 'year' axis)
# ============================================================
for model in MODELS:
    ds = xr.open_dataset(AMOC_FILE_TPL.format(model=model))
    keep = [v for v in ds.data_vars if v.startswith("AMOC_")]
    ds = ds[keep]
    if "time" in ds.dims:
        ds = ds.assign_coords(year=("time", extract_years(ds["time"].values))).swap_dims(time="year").drop_vars("time")
    if "member" in ds.coords:
        ds = ds.assign_coords(member=ds["member"].astype(str))
    out = os.path.join(OUT_DIR, f"{model}_amoc.nc")
    ds.load().to_netcdf(out)
    print("✓", out, keep)

In [ ]:
# Size check
tot = 0
for f in sorted(glob.glob(os.path.join(OUT_DIR, "*.nc"))):
    s = os.path.getsize(f) / 1e6; tot += s
    print(f"{os.path.basename(f):45s} {s:8.1f} MB")
print(f"{'TOTAL':45s} {tot:8.1f} MB")